In [ ]:
#Week 4-4
#Bolortulga Seded
#24045487
#Feature Encoding -Employees dataset

In [5]:
import pandas as pd
import numpy as np
import sklearn.preprocessing
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer

df = pd.read_csv('employees.csv')
print(df.shape)
print(df.dtypes)
df.head()

(1000, 8)
First Name               str
Gender                   str
Start Date               str
Last Login Time          str
Salary                 int64
Bonus %              float64
Senior Management     object
Team                     str
dtype: object


,First Name,Gender,Start Date,Last Login Time,Salary,Bonus %,Senior Management,Team
0,Douglas,Male,8/6/1993,12:42 PM,97308,6.945,True,Marketing
1,Thomas,Male,3/31/1996,6:53 AM,61933,4.170,True,NaN
2,Maria,Female,4/23/1993,11:17 AM,130590,11.858,False,Finance
3,Jerry,Male,3/4/2005,1:00 PM,138705,9.340,True,Finance
4,Larry,Male,1/24/1998,4:47 PM,101004,1.389,True,Client Services


In [8]:
cat_cols = df.select_dtypes(include='str').columns.tolist()
print("Categorical columns:", cat_cols)

for col in cat_cols:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(df[col].value_counts().head())

Categorical columns: ['First Name', 'Gender', 'Start Date', 'Last Login Time', 'Team']

First Name: 200 unique values
First Name
Marilyn    11
Jeremy     10
Todd       10
Barbara    10
Cynthia     9
Name: count, dtype: int64

Gender: 2 unique values
Gender
Female    431
Male      424
Name: count, dtype: int64

Start Date: 972 unique values
Start Date
3/31/1996     2
1/26/2005     2
2/16/2009     2
12/17/2011    2
10/30/2004    2
Name: count, dtype: int64

Last Login Time: 720 unique values
Last Login Time
1:35 PM     5
1:00 PM     4
1:08 AM     4
11:25 AM    4
11:57 PM    4
Name: count, dtype: int64

Team: 10 unique values
Team
Client Services         106
Finance                 102
Business Development    101
Marketing                98
Product                  95
Name: count, dtype: int64


In [13]:
le = LabelEncoder()
df_label = df.copy()

for col in ['Gender', 'Senior Management']:
    df_label[col + '_encoded'] = le.fit_transform(df_label[col].astype(str))
    print(f"\n{col} mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

print(df_label[['Gender', 'Gender_encoded', 
                'Senior Management', 'Senior Management_encoded']].head())


Gender mapping: {'Female': np.int64(0), 'Male': np.int64(1), nan: np.int64(2)}

Senior Management mapping: {'False': np.int64(0), 'True': np.int64(1), nan: np.int64(2)}
   Gender  Gender_encoded Senior Management  Senior Management_encoded
0    Male               1              True                          1
1    Male               1              True                          1
2  Female               0             False                          0
3    Male               1              True                          1
4    Male               1              True                          1


In [14]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_array = ohe.fit_transform(df[['Gender','Team']].fillna('Unknown'))

print("OHE feature names:", ohe.get_feature_names_out(['Gender', 'Team']))
ohe_df = pd.DataFrame(ohe_array, 
                      columns=ohe.get_feature_names_out(['Gender', 'Team']))
df_ohe = pd.concat([df.reset_index(drop=True), ohe_df], axis=1)
print(df_ohe.head())
print("Shape after OHE:", df_ohe.shape)

OHE feature names: ['Gender_Female' 'Gender_Male' 'Gender_Unknown'
 'Team_Business Development' 'Team_Client Services' 'Team_Distribution'
 'Team_Engineering' 'Team_Finance' 'Team_Human Resources' 'Team_Legal'
 'Team_Marketing' 'Team_Product' 'Team_Sales' 'Team_Unknown']
  First Name  Gender Start Date Last Login Time  Salary  Bonus %  \
0    Douglas    Male   8/6/1993        12:42 PM   97308    6.945   
1     Thomas    Male  3/31/1996         6:53 AM   61933    4.170   
2      Maria  Female  4/23/1993        11:17 AM  130590   11.858   
3      Jerry    Male   3/4/2005         1:00 PM  138705    9.340   
4      Larry    Male  1/24/1998         4:47 PM  101004    1.389   

  Senior Management             Team  Gender_Female  Gender_Male  ...  \
0              True        Marketing            0.0          1.0  ...   
1              True              NaN            0.0          1.0  ...   
2             False          Finance            1.0          0.0  ...   
3              True        

In [17]:
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df_ord = df.copy()
df_ord['Team_encoded'] = oe.fit_transform(df_ord[['Team']].fillna('Unknown'))

print(df_ord[['Team', 'Team_encoded']].drop_duplicates()
              .sort_values('Team_encoded'))

                    Team  Team_encoded
9   Business Development           0.0
4        Client Services           1.0
40          Distribution           2.0
8            Engineering           3.0
2                Finance           4.0
12       Human Resources           5.0
5                  Legal           6.0
0              Marketing           7.0
6                Product           8.0
13                 Sales           9.0
1                    NaN          10.0


In [20]:
nominal_features = ['Gender', 'Team']
ordinal_features = ['Senior Management']

preprocessor = ColumnTransformer(
    transformers=[
        ('nom', OneHotEncoder(sparse_output=False,  
                             handle_unknown='ignore'), nominal_features),
        ('ord', OrdinalEncoder(handle_unknown='use_encoded_value',
                               unknown_value=-1), ordinal_features)
    ],
    remainder='passthrough'
)
X = df[['Gender', 'Team', 'Senior Management' ,'Salary', 'Bonus %']]
X_prepared = preprocessor.fit_transform(X)
print("Transformed shape:", X_prepared.shape)
    
        

Transformed shape: (1000, 17)


In [21]:
final_df = pd.DataFrame(
    X_prepared,
    columns=list(preprocessor.get_feature_names_out())
)
print(final_df.head())
print("Final shape:", final_df.shape)

   nom__Gender_Female  nom__Gender_Male  nom__Gender_nan  \
0                 0.0               1.0              0.0   
1                 0.0               1.0              0.0   
2                 1.0               0.0              0.0   
3                 0.0               1.0              0.0   
4                 0.0               1.0              0.0   

   nom__Team_Business Development  nom__Team_Client Services  \
0                             0.0                        0.0   
1                             0.0                        0.0   
2                             0.0                        0.0   
3                             0.0                        0.0   
4                             0.0                        1.0   

   nom__Team_Distribution  nom__Team_Engineering  nom__Team_Finance  \
0                     0.0                    0.0                0.0   
1                     0.0                    0.0                0.0   
2                     0.0                